In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import GridSearchCV,train_test_split,cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

In [2]:
!pip install pyxlsb

In [3]:
!pip install xlrd

In [4]:
import pandas as pd
import numpy as np
import warnings

# Suppress the specific pandas warnings
warnings.filterwarnings('ignore', message='invalid value encountered in greater')
warnings.filterwarnings('ignore', message='invalid value encountered in less')

# Or suppress all RuntimeWarnings from pandas formatting
warnings.filterwarnings('ignore', category=RuntimeWarning, module='pandas.io.formats.format')

# Now read your data
df = pd.read_excel('rawData.xlsx')
df = df[df['Sign Type Broad Category'] == 'Halo lit Chanel letter sign']
df = df.reset_index(drop=True)


print("Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Check for data quality issues that might cause these warnings
print("\n" + "="*50)
print("DATA QUALITY CHECK:")

# Check for missing values
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Columns with missing values:")
missing_cols = df.isnull().sum()
for col in missing_cols[missing_cols > 0].index:
    print(f"  - {col}: {missing_cols[col]} missing")

# Check data types
print(f"\nData types:")
for col, dtype in df.dtypes.items():
    print(f"  - {col}: {dtype}")

# Check for mixed data types in numeric columns
print(f"\nChecking for mixed data types:")
for col in df.columns:
    if df[col].dtype == 'object':  # String columns might contain mixed types
        # Try to identify if it should be numeric
        sample_values = df[col].dropna().astype(str).str.strip()
        if len(sample_values) > 0:
            # Check if values look numeric
            numeric_pattern = sample_values.str.match(r'^-?\d+\.?\d*$')
            if numeric_pattern.any():
                numeric_count = numeric_pattern.sum()
                total_count = len(sample_values)
                if numeric_count > total_count * 0.5:  # More than 50% numeric
                    print(f"  - {col}: Appears to be numeric but stored as object ({numeric_count}/{total_count} numeric)")

# Safe display function that handles problematic data
def safe_display(df, n_rows=5):
    """Display dataframe without triggering formatting warnings"""
    try:
        # Create a copy for display
        display_df = df.head(n_rows).copy()
        
        # Replace problematic values for display
        for col in display_df.columns:
            if display_df[col].dtype in ['float64', 'int64']:
                # Replace inf and -inf with string representations
                display_df[col] = display_df[col].replace([np.inf, -np.inf], ['inf', '-inf'])
        
        return display_df
    except Exception as e:
        print(f"Display error: {e}")
        return df.head(n_rows)

print(f"\n" + "="*50)
print("FIRST FEW ROWS (safe display):")
display_data = safe_display(df)
print(display_data)

# Clean up numeric columns if needed
print(f"\n" + "="*50)
print("CLEANING NUMERIC COLUMNS:")

numeric_cols = []
for col in df.columns:
    if 'price' in col.lower() or 'cost' in col.lower() or 'amount' in col.lower() or 'width' in col.lower() or 'height' in col.lower() or 'area' in col.lower():
        numeric_cols.append(col)

if numeric_cols:
    print(f"Found potential numeric columns: {numeric_cols}")
    
    for col in numeric_cols:
        if col in df.columns:
            print(f"\nCleaning column: {col}")
            original_type = df[col].dtype
            
            try:
                # Convert to numeric, coercing errors to NaN
                df[col] = pd.to_numeric(df[col], errors='coerce')
                print(f"  - Converted from {original_type} to {df[col].dtype}")
                print(f"  - NaN values after conversion: {df[col].isnull().sum()}")
                
            except Exception as e:
                print(f"  - Could not convert {col}: {e}")

# Final summary
print(f"\n" + "="*50)
print("FINAL DATA SUMMARY:")
print(f"Shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print(f"Total missing values: {df.isnull().sum().sum()}")

# Display basic statistics for numeric columns
numeric_columns = df.select_dtypes(include=[np.number]).columns
if len(numeric_columns) > 0:
    print(f"\nNumeric columns summary:")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        print(df[numeric_columns].describe())

print("\nWarnings should now be suppressed!")
print("\nYour dataframe is ready to use: 'df'")

Data loaded successfully!
Shape: (1379, 19)
Columns: ['Order ID', 'Order Date', 'Order Added in Month Tab', 'Account Name', 'Sign Type', 'Sign Type Broad Category', 'Month (AT)', 'Sign Width (in)', 'Sign Height (in)', 'Selling Price (USD)', 'Withdrawal Amount (USD)', 'Project Name', 'Status', 'Production Line', 'BOM - Material Cost (PKR) - Calculated By Ali Hassan', '📙 BOM - Production Cost (USD)', '📙 BOM - Shipping Cost (USD)', 'Sign Area (sq.ft)', 'Length of Curve (m)']

DATA QUALITY CHECK:
Total missing values: 3319
Columns with missing values:
  - Order Date: 1 missing
  - Sign Type: 1 missing
  - Sign Width (in): 4 missing
  - Sign Height (in): 4 missing
  - Withdrawal Amount (USD): 1379 missing
  - Project Name: 6 missing
  - Status: 231 missing
  - BOM - Material Cost (PKR) - Calculated By Ali Hassan: 97 missing
  - 📙 BOM - Production Cost (USD): 1 missing
  - 📙 BOM - Shipping Cost (USD): 216 missing
  - Length of Curve (m): 1379 missing

Data types:
  - Order ID: object
  - Ord

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1379 entries, 0 to 1378
Data columns (total 19 columns):
 #   Column                                                Non-Null Count  Dtype         
---  ------                                                --------------  -----         
 0   Order ID                                              1379 non-null   object        
 1   Order Date                                            1378 non-null   datetime64[ns]
 2   Order Added in Month Tab                              1379 non-null   datetime64[ns]
 3   Account Name                                          1379 non-null   object        
 4   Sign Type                                             1378 non-null   object        
 5   Sign Type Broad Category                              1379 non-null   object        
 6   Month (AT)                                            1379 non-null   datetime64[ns]
 7   Sign Width (in)                                       1375 non-null   float64 

In [6]:
print(f'The number of rows are {df.shape[0]} and columns are {df.shape[1]}')

The number of rows are 1379 and columns are 19


In [7]:
df.columns

Index(['Order ID', 'Order Date', 'Order Added in Month Tab', 'Account Name',
       'Sign Type', 'Sign Type Broad Category', 'Month (AT)',
       'Sign Width (in)', 'Sign Height (in)', 'Selling Price (USD)',
       'Withdrawal Amount (USD)', 'Project Name', 'Status', 'Production Line',
       'BOM - Material Cost (PKR) - Calculated By Ali Hassan',
       '📙 BOM - Production Cost (USD)', '📙 BOM - Shipping Cost (USD)',
       'Sign Area (sq.ft)', 'Length of Curve (m)'],
      dtype='object')

In [8]:
df.head(5)

,Order ID,Order Date,Order Added in Month Tab,Account Name,Sign Type,Sign Type Broad Category,Month (AT),Sign Width (in),Sign Height (in),Selling Price (USD),Withdrawal Amount (USD),Project Name,Status,Production Line,BOM - Material Cost (PKR) - Calculated By Ali Hassan,📙 BOM - Production Cost (USD),📙 BOM - Shipping Cost (USD),Sign Area (sq.ft),Length of Curve (m)
0,BS-ET-7833 A,2024-12-30,2024-12-30,ArtfulAdornmentz,Metal - With Backlit,Halo lit Chanel letter sign,2025-01-01,72.0,10.9,550.0,NaN,ETSY Project,Shipped,Business Sign,NaN,207.24,205.81,5,NaN
1,BS-ET-7833 B,2024-12-30,2024-12-30,ArtfulAdornmentz,Metal - With Backlit,Halo lit Chanel letter sign,2025-01-01,32.4,36.0,700.0,NaN,ETSY Project,Shipped,Business Sign,NaN,370.56,205.81,8,NaN
2,BS-SM-7798 B,2024-12-24,2025-01-01,Signmakerz-Ads,3D Metal with UV Printed,Halo lit Chanel letter sign,2025-01-01,90.0,36.0,1251.0,NaN,Google Ads Project,Shipped,Business Sign,NaN,268.85,185.47,23,NaN
3,BS-SI-7829,2024-12-30,2025-01-01,Signs INC-Ads,Metal - With Backlit,Halo lit Chanel letter sign,2025-01-01,20.0,23.0,567.0,NaN,Google Ads Project,Shipped,Business Sign,NaN,106.07,99.57,3,NaN
4,BS-SI-7802,2024-12-24,2025-01-01,Signs INC-Ads,3D Metal with UV Printed,Halo lit Chanel letter sign,2025-01-01,30.0,24.0,525.0,NaN,Google Ads Project,Shipped,Business Sign,NaN,140.55,125.45,5,NaN


In [9]:
df.isnull().sum()

Order ID                                                   0
Order Date                                                 1
Order Added in Month Tab                                   0
Account Name                                               0
Sign Type                                                  1
Sign Type Broad Category                                   0
Month (AT)                                                 0
Sign Width (in)                                            4
Sign Height (in)                                           4
Selling Price (USD)                                        0
Withdrawal Amount (USD)                                 1379
Project Name                                               6
Status                                                   231
Production Line                                            0
BOM - Material Cost (PKR) - Calculated By Ali Hassan    1379
📙 BOM - Production Cost (USD)                              1
📙 BOM - Shipping Cost (U

In [10]:
df.drop_duplicates(inplace=True)

In [11]:
df.describe(include='all')

,Order ID,Order Date,Order Added in Month Tab,Account Name,Sign Type,Sign Type Broad Category,Month (AT),Sign Width (in),Sign Height (in),Selling Price (USD),Withdrawal Amount (USD),Project Name,Status,Production Line,BOM - Material Cost (PKR) - Calculated By Ali Hassan,📙 BOM - Production Cost (USD),📙 BOM - Shipping Cost (USD),Sign Area (sq.ft),Length of Curve (m)
count,1379,1378,1379,1379,1378,1379,1379,1375.000000,1375.000000,1379.000000,0.0,1373,1148,1379,0.0,1378.000000,1163.000000,1379.000000,0.0
unique,1377,NaN,NaN,52,12,1,NaN,NaN,NaN,NaN,NaN,6,52,1,NaN,NaN,NaN,NaN,NaN
top,BS-SI-10967,NaN,NaN,Signmakerz-Ads,Metal - With Backlit,Halo lit Chanel letter sign,NaN,NaN,NaN,NaN,NaN,Google Ads Project,Shipped,Business Sign,NaN,NaN,NaN,NaN,NaN
freq,2,NaN,NaN,276,1024,1379,NaN,NaN,NaN,NaN,NaN,705,946,1379,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-04-08 11:06:42.322206208,2025-04-16 03:44:30.630891776,NaN,NaN,NaN,2025-04-02 03:54:57.171863552,51.806327,28.462909,975.259608,NaN,NaN,NaN,NaN,NaN,304.405842,174.804488,10.242204,NaN
min,NaN,2022-11-29 00:00:00,2022-11-29 00:00:00,NaN,NaN,NaN,2022-11-01 00:00:00,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN
25%,NaN,2025-02-17 00:00:00,2025-02-24 00:00:00,NaN,NaN,NaN,2025-02-01 00:00:00,34.200000,15.000000,450.000000,NaN,NaN,NaN,NaN,NaN,144.880000,87.050000,4.000000,NaN
50%,NaN,2025-04-03 00:00:00,2025-04-08 00:00:00,NaN,NaN,NaN,2025-04-01 00:00:00,48.000000,24.000000,735.000000,NaN,NaN,NaN,NaN,NaN,237.315000,137.200000,8.000000,NaN
75%,NaN,2025-05-29 00:00:00,2025-06-08 00:00:00,NaN,NaN,NaN,2025-06-01 00:00:00,60.000000,36.000000,1197.000000,NaN,NaN,NaN,NaN,NaN,379.765000,219.880000,13.000000,NaN
max,NaN,2025-12-31 00:00:00,2025-08-30 00:00:00,NaN,NaN,NaN,2025-08-01 00:00:00,319.200000,350.000000,9874.000000,NaN,NaN,NaN,NaN,NaN,4453.310000,2207.390000,162.000000,NaN


In [12]:
df['depth'] = 1

In [13]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from skopt import BayesSearchCV
import xgboost as xgb
import lightgbm as lgb
import pickle
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# =====================
# SETUP
# =====================

SIGN_TYPE = "halolit_channel_letters"   # change this to flatcut_letters / backlit_metal_signs
MODEL_DIR = os.path.join("model", SIGN_TYPE)
os.makedirs(MODEL_DIR, exist_ok=True)


# Rename columns
df_clean = df.copy()
df_clean = df_clean.rename(columns={
    'Sign Width (in)': 'width',
    'Sign Height (in)': 'height',
    'Depth': 'depth',
    '📙 BOM - Shipping Cost (USD)' : 'shipping_cost',
    'Sign Area (sq.ft)' : 'Sign Area(in)'
})

print("Renamed columns successfully!")

# =====================
# DATA PREP
# =====================
feature_columns = ['width', 'height', 'depth', 'Sign Area(in)', 'shipping_cost']
data_for_imputation = df_clean[feature_columns].copy()

print("\nApplying KNN Imputation...")
knn_imputer = KNNImputer(n_neighbors=4)
data_imputed = knn_imputer.fit_transform(data_for_imputation)
df_imputed = pd.DataFrame(data_imputed, columns=feature_columns, index=df_clean.index)

# Features / Target
X = df_imputed[['width', 'height', 'depth', 'Sign Area(in)']].copy()
y = df_imputed['shipping_cost'].copy()

# Scale + Split
scaler = MaxAbsScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# =====================
# MODEL TRAINING
# =====================
models_results = {}

def train_and_eval(name, estimator, param_grid, n_iter=50):
    """Train model with BayesSearchCV safely"""
    try:
        print(f"\n{name} with Bayesian Optimization:")
        search = BayesSearchCV(
            estimator=estimator,
            search_spaces=param_grid,
            n_iter=n_iter,
            cv=5,
            scoring='r2',
            random_state=42,
            n_jobs=-1
        )
        search.fit(X_train, y_train)
        y_pred = search.best_estimator_.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        print(f"{name} Test Metrics → MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.3f}")
        models_results[name] = {
            'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2, 'model': search.best_estimator_
        }
    except Exception as e:
        print(f"⚠️ {name} failed: {e}")

# =====================
# TRAIN ALL MODELS
# =====================
train_and_eval("Random Forest", RandomForestRegressor(random_state=42), {
    'n_estimators': (10, 200), 'max_depth': (3, 20),
    'min_samples_split': (2, 15), 'min_samples_leaf': (1, 10),
    'max_features': ['sqrt', 'log2', None]
})

train_and_eval("Gradient Boosting", GradientBoostingRegressor(random_state=42), {
    'n_estimators': (50, 300), 'max_depth': (3, 10),
    'learning_rate': (0.01, 0.3), 'min_samples_split': (2, 20),
    'min_samples_leaf': (1, 10), 'subsample': (0.8, 1.0)
})

train_and_eval("SVR", SVR(kernel='rbf'), {
    'C': (1, 1000), 'gamma': (0.001, 1), 'epsilon': (0.01, 1)
})

train_and_eval("Decision Tree", DecisionTreeRegressor(random_state=42), {
    'max_depth': (3, 20), 'min_samples_split': (2, 20),
    'min_samples_leaf': (1, 10), 'max_features': ['sqrt', 'log2', None]
})

train_and_eval("XGBoost", xgb.XGBRegressor(random_state=42, eval_metric='rmse'), {
    'n_estimators': (50, 300), 'max_depth': (3, 10),
    'learning_rate': (0.01, 0.3), 'subsample': (0.8, 1.0),
    'colsample_bytree': (0.8, 1.0), 'reg_alpha': (0, 1), 'reg_lambda': (0, 1)
})

train_and_eval("LightGBM", lgb.LGBMRegressor(random_state=42, verbose=-1), {
    'n_estimators': (50, 300), 'max_depth': (3, 10),
    'learning_rate': (0.01, 0.3), 'subsample': (0.8, 1.0),
    'colsample_bytree': (0.8, 1.0), 'reg_alpha': (0, 1),
    'reg_lambda': (0, 1), 'num_leaves': (10, 100)
})

train_and_eval("Extra Trees", ExtraTreesRegressor(random_state=42), {
    'n_estimators': (10, 200), 'max_depth': (3, 20),
    'min_samples_split': (2, 15), 'min_samples_leaf': (1, 10),
    'max_features': ['sqrt', 'log2', None]
})

train_and_eval("Ridge", Ridge(random_state=42), {
    'alpha': (0.1, 100)
}, n_iter=30)

train_and_eval("ElasticNet", ElasticNet(random_state=42), {
    'alpha': (0.1, 10), 'l1_ratio': (0.1, 0.9)
}, n_iter=30)

# =====================
# BEST MODEL SELECTION
# =====================
if models_results:
    best_model_name = max(models_results.keys(), key=lambda x: models_results[x]['R2'])
    best_model = models_results[best_model_name]['model']
    print(f"\n✅ Best Model: {best_model_name} (R² = {models_results[best_model_name]['R2']:.3f})")
else:
    best_model_name, best_model = None, None
    print("\n❌ No models trained successfully!")

# =====================
# SAVE MODELS
# =====================
if models_results:
    high_accuracy_models = {n: r for n, r in models_results.items() if r['R2'] > 0.7}
    to_save = high_accuracy_models if high_accuracy_models else dict(list(models_results.items())[:3])

    for model_name, results in to_save.items():
        joblib.dump(results['model'], f"{MODEL_DIR}/{model_name.replace(' ', '_').lower()}_model.joblib")
        with open(f"{MODEL_DIR}/{model_name.replace(' ', '_').lower()}_model.pkl", 'wb') as f:
            pickle.dump(results['model'], f)

    if best_model:
        joblib.dump(best_model, f"{MODEL_DIR}/best_model_{best_model_name.replace(' ', '_').lower()}.joblib")

    joblib.dump(scaler, f"{MODEL_DIR}/scaler.joblib")
    with open(f"{MODEL_DIR}/feature_names.pkl", 'wb') as f:
        pickle.dump(['width', 'height', 'depth', 'Selling Price (USD)', 'Sign Area (sq.ft)'], f)
    with open(f"{MODEL_DIR}/model_results.pkl", 'wb') as f:
        pickle.dump(models_results, f)

    print(f"\n📂 Saved models in {MODEL_DIR}/")
    for file in os.listdir(MODEL_DIR):
        print(" -", file)

print("\n🚀 Mission accomplished!")


Renamed columns successfully!

Applying KNN Imputation...
Train: (1103, 4), Test: (276, 4)

Random Forest with Bayesian Optimization:
Random Forest Test Metrics → MAE: 57.29, RMSE: 106.22, R²: 0.502

Gradient Boosting with Bayesian Optimization:
Gradient Boosting Test Metrics → MAE: 59.62, RMSE: 110.67, R²: 0.459

SVR with Bayesian Optimization:
SVR Test Metrics → MAE: 58.53, RMSE: 111.75, R²: 0.449

Decision Tree with Bayesian Optimization:
Decision Tree Test Metrics → MAE: 62.33, RMSE: 115.17, R²: 0.414

XGBoost with Bayesian Optimization:
XGBoost Test Metrics → MAE: 59.65, RMSE: 110.91, R²: 0.457

LightGBM with Bayesian Optimization:
LightGBM Test Metrics → MAE: 59.33, RMSE: 111.20, R²: 0.454

Extra Trees with Bayesian Optimization:
Extra Trees Test Metrics → MAE: 56.44, RMSE: 106.69, R²: 0.497

Ridge with Bayesian Optimization:
Ridge Test Metrics → MAE: 57.40, RMSE: 103.69, R²: 0.525

ElasticNet with Bayesian Optimization:
ElasticNet Test Metrics → MAE: 69.56, RMSE: 117.83, R²: 0.3